In [0]:
from pyspark.sql import functions as F

appointments_bronze = spark.table(
    "healthcare.default.bronze_appointments"
)

patients_bronze = spark.table(
    "healthcare.default.bronze_patients"
)

doctors_bronze = spark.table(
    "healthcare.default.bronze_doctors"
)

print("Appointments:", appointments_bronze.count())
print("Patients:", patients_bronze.count())
print("Doctors:", doctors_bronze.count())

Appointments: 200
Patients: 50
Doctors: 10


In [0]:
display(appointments_bronze)

appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
A001,P034,D009,2023-08-09,2026-08-10T15:15:00.000Z,Therapy,Scheduled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,39129462e5fd1bff89a54b86c6bd10cc636bf78ed464a8c833809f18a05ac71c,BRONZE
A002,P032,D004,2023-06-09,2026-08-10T14:30:00.000Z,Therapy,No-show,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,94e5618f556eda537cbd89100999833711c6ea8d3064b7cf71728df6c830bb0a,BRONZE
A003,P048,D004,2023-06-28,2026-08-10T08:00:00.000Z,Consultation,Cancelled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f15b73ac6f2f662ecaa2e0cf1ade2a405df630fbec5a0845f0e67a5d387a989a,BRONZE
A004,P025,D006,2023-09-01,2026-08-10T09:15:00.000Z,Consultation,Cancelled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,40c3a0117e75056382aff06888c18b72e193d7c745f3f3a2cf95335b170631bc,BRONZE
A005,P040,D003,2023-07-06,2026-08-10T12:45:00.000Z,Emergency,No-show,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9ee8014971d675609a0d868f8bcc68972d46e04991d189beb4daaa9ff3a7500c,BRONZE
A006,P045,D006,2023-06-19,2026-08-10T16:15:00.000Z,Checkup,Scheduled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9df730a3521ae05c83de230fa69c9d1575dd4e4361d4c0aa22ed2d9fb1c60866,BRONZE
A007,P001,D007,2023-04-09,2026-08-10T10:30:00.000Z,Consultation,Scheduled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,3eb0a8c591025b2af9fe59b3a022f13bb637d30d693deead97866fde92e3f433,BRONZE
A008,P016,D010,2023-05-24,2026-08-10T08:45:00.000Z,Consultation,Cancelled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,57f93870c1c3feba455ba046b386c5b2a605bbf4e300d10a6a9237262f84a617,BRONZE
A009,P039,D010,2023-03-05,2026-08-10T13:45:00.000Z,Follow-up,Scheduled,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,21e65c5fe3199f0cdfddf55c8eeb6d4a39b261683c40ec08a367ecfbb9f489f6,BRONZE
A010,P005,D003,2023-01-13,2026-08-10T15:30:00.000Z,Therapy,Completed,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f806fc8f83616deb6e5e5f6938c43a911d63205973babce3d539d370284b8521,BRONZE


In [0]:
duplicate_appointments = (
    appointments_bronze
    .groupBy("appointment_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate appointment IDs:",
    duplicate_appointments.count()
)

display(duplicate_appointments)

Duplicate appointment IDs: 0


appointment_id,count


In [0]:
appointment_nulls = appointments_bronze.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in appointments_bronze.columns
])

display(appointment_nulls)

appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
display(
    appointments_bronze
    .groupBy("status")
    .count()
    .orderBy("status")
)

status,count
Cancelled,51
Completed,46
No-show,52
Scheduled,51


In [0]:
invalid_patients = (
    appointments_bronze
    .join(
        patients_bronze.select("patient_id").distinct(),
        on="patient_id",
        how="left_anti"
    )
)

print(
    "Appointments with invalid patient_id:",
    invalid_patients.count()
)

display(invalid_patients)

Appointments with invalid patient_id: 0


patient_id,appointment_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
invalid_doctors = (
    appointments_bronze
    .join(
        doctors_bronze.select("doctor_id").distinct(),
        on="doctor_id",
        how="left_anti"
    )
)

print(
    "Appointments with invalid doctor_id:",
    invalid_doctors.count()
)

display(invalid_doctors)

Appointments with invalid doctor_id: 0


doctor_id,appointment_id,patient_id,appointment_date,appointment_time,reason_for_visit,status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
appointments_bronze.printSchema()

root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- appointment_time: timestamp (nullable = true)
 |-- reason_for_visit: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _source_id: string (nullable = true)
 |-- _source_name: string (nullable = true)
 |-- _source_file_name: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- _record_hash: string (nullable = true)
 |-- _layer: string (nullable = true)



In [0]:
display(
    appointments_bronze
    .groupBy("reason_for_visit")
    .count()
    .orderBy("reason_for_visit")
)

reason_for_visit,count
Checkup,45
Consultation,43
Emergency,29
Follow-up,41
Therapy,42


In [0]:
appointment_date_check = appointments_bronze.select(
    F.min("appointment_date").alias("earliest_appointment"),
    F.max("appointment_date").alias("latest_appointment")
)

display(appointment_date_check)

earliest_appointment,latest_appointment
2023-01-01,2023-12-30


In [0]:
display(
    appointments_bronze.select(
        "appointment_id",
        "appointment_date",
        "appointment_time"
    ).limit(20)
)

appointment_id,appointment_date,appointment_time
A001,2023-08-09,2026-08-10T15:15:00.000Z
A002,2023-06-09,2026-08-10T14:30:00.000Z
A003,2023-06-28,2026-08-10T08:00:00.000Z
A004,2023-09-01,2026-08-10T09:15:00.000Z
A005,2023-07-06,2026-08-10T12:45:00.000Z
A006,2023-06-19,2026-08-10T16:15:00.000Z
A007,2023-04-09,2026-08-10T10:30:00.000Z
A008,2023-05-24,2026-08-10T08:45:00.000Z
A009,2023-03-05,2026-08-10T13:45:00.000Z
A010,2023-01-13,2026-08-10T15:30:00.000Z


In [0]:
# Clean and standardize appointment data

silver_appointments = (
    appointments_bronze

    # Clean text fields
    .withColumn(
        "reason_for_visit",
        F.trim(F.col("reason_for_visit"))
    )

    .withColumn(
        "status",
        F.initcap(F.trim(F.col("status")))
    )

    # Extract only the time from the timestamp
    .withColumn(
        "appointment_time_clean",
        F.date_format(
            F.col("appointment_time"),
            "HH:mm:ss"
        )
    )

    # Validation flags
    .withColumn(
        "valid_appointment_id",
        F.col("appointment_id").isNotNull()
    )

    .withColumn(
        "valid_patient_id",
        F.col("patient_id").isNotNull()
    )

    .withColumn(
        "valid_doctor_id",
        F.col("doctor_id").isNotNull()
    )

    .withColumn(
        "valid_appointment_date",
        F.col("appointment_date").isNotNull()
    )

    # Keep only valid records
    .filter(
        F.col("valid_appointment_id")
        & F.col("valid_patient_id")
        & F.col("valid_doctor_id")
        & F.col("valid_appointment_date")
    )

    # Select final Silver columns
    .select(
        "appointment_id",
        "patient_id",
        "doctor_id",
        "appointment_date",

        F.col("appointment_time_clean")
         .alias("appointment_time"),

        "reason_for_visit",
        "status",

        "valid_appointment_id",
        "valid_patient_id",
        "valid_doctor_id",
        "valid_appointment_date",

        # Pipeline lineage
        "_batch_id",
        "_source_id",
        "_source_name",
        "_source_file_name",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_record_hash"
    )
)

print(
    "Silver appointment records:",
    silver_appointments.count()
)

display(silver_appointments)

Silver appointment records: 200


appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,valid_appointment_id,valid_patient_id,valid_doctor_id,valid_appointment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash
A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,39129462e5fd1bff89a54b86c6bd10cc636bf78ed464a8c833809f18a05ac71c
A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,94e5618f556eda537cbd89100999833711c6ea8d3064b7cf71728df6c830bb0a
A003,P048,D004,2023-06-28,08:00:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f15b73ac6f2f662ecaa2e0cf1ade2a405df630fbec5a0845f0e67a5d387a989a
A004,P025,D006,2023-09-01,09:15:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,40c3a0117e75056382aff06888c18b72e193d7c745f3f3a2cf95335b170631bc
A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9ee8014971d675609a0d868f8bcc68972d46e04991d189beb4daaa9ff3a7500c
A006,P045,D006,2023-06-19,16:15:00,Checkup,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9df730a3521ae05c83de230fa69c9d1575dd4e4361d4c0aa22ed2d9fb1c60866
A007,P001,D007,2023-04-09,10:30:00,Consultation,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,3eb0a8c591025b2af9fe59b3a022f13bb637d30d693deead97866fde92e3f433
A008,P016,D010,2023-05-24,08:45:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,57f93870c1c3feba455ba046b386c5b2a605bbf4e300d10a6a9237262f84a617
A009,P039,D010,2023-03-05,13:45:00,Follow-up,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,21e65c5fe3199f0cdfddf55c8eeb6d4a39b261683c40ec08a367ecfbb9f489f6
A010,P005,D003,2023-01-13,15:30:00,Therapy,Completed,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f806fc8f83616deb6e5e5f6938c43a911d63205973babce3d539d370284b8521


In [0]:
silver_appointments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthcare.default.silver_appointments"
    )

print("silver_appointments created successfully.")

silver_appointments created successfully.


In [0]:
display(
    spark.table(
        "healthcare.default.silver_appointments"
    )
)

appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,valid_appointment_id,valid_patient_id,valid_doctor_id,valid_appointment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash
A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,39129462e5fd1bff89a54b86c6bd10cc636bf78ed464a8c833809f18a05ac71c
A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,94e5618f556eda537cbd89100999833711c6ea8d3064b7cf71728df6c830bb0a
A003,P048,D004,2023-06-28,08:00:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f15b73ac6f2f662ecaa2e0cf1ade2a405df630fbec5a0845f0e67a5d387a989a
A004,P025,D006,2023-09-01,09:15:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,40c3a0117e75056382aff06888c18b72e193d7c745f3f3a2cf95335b170631bc
A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9ee8014971d675609a0d868f8bcc68972d46e04991d189beb4daaa9ff3a7500c
A006,P045,D006,2023-06-19,16:15:00,Checkup,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9df730a3521ae05c83de230fa69c9d1575dd4e4361d4c0aa22ed2d9fb1c60866
A007,P001,D007,2023-04-09,10:30:00,Consultation,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,3eb0a8c591025b2af9fe59b3a022f13bb637d30d693deead97866fde92e3f433
A008,P016,D010,2023-05-24,08:45:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,57f93870c1c3feba455ba046b386c5b2a605bbf4e300d10a6a9237262f84a617
A009,P039,D010,2023-03-05,13:45:00,Follow-up,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,21e65c5fe3199f0cdfddf55c8eeb6d4a39b261683c40ec08a367ecfbb9f489f6
A010,P005,D003,2023-01-13,15:30:00,Therapy,Completed,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f806fc8f83616deb6e5e5f6938c43a911d63205973babce3d539d370284b8521


In [0]:
display(
    spark.table(
        "healthcare.default.silver_appointments"
    )
    .groupBy("status")
    .count()
    .orderBy("status")
)

status,count
Cancelled,51
Completed,46
No-show,52
Scheduled,51


In [0]:
display(
    spark.table(
        "healthcare.default.silver_appointments"
    )
    .select(
        "appointment_id",
        "appointment_date",
        "appointment_time",
        "reason_for_visit",
        "status"
    )
    .limit(20)
)

appointment_id,appointment_date,appointment_time,reason_for_visit,status
A001,2023-08-09,15:15:00,Therapy,Scheduled
A002,2023-06-09,14:30:00,Therapy,No-show
A003,2023-06-28,08:00:00,Consultation,Cancelled
A004,2023-09-01,09:15:00,Consultation,Cancelled
A005,2023-07-06,12:45:00,Emergency,No-show
A006,2023-06-19,16:15:00,Checkup,Scheduled
A007,2023-04-09,10:30:00,Consultation,Scheduled
A008,2023-05-24,08:45:00,Consultation,Cancelled
A009,2023-03-05,13:45:00,Follow-up,Scheduled
A010,2023-01-13,15:30:00,Therapy,Completed
